In [ ]:
# Databricks notebook source
# COMMAND ----------
# %md
# # Customer 360 ETL Process
# This notebook performs an ETL process to create a comprehensive customer profile by ingesting, transforming, and enriching data from various sources.

# COMMAND ----------
#
# Import necessary libraries
import logging
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# COMMAND ----------
#
# Function to load data from CSV files into DataFrames
def load_data():
    logger.info("Loading data from CSV files into DataFrames.")
    policy_df = spark.read.csv("dbfs:/mnt/data/policy.csv", header=True, inferSchema=True)
    claims_df = spark.read.csv("dbfs:/mnt/data/claims.csv", header=True, inferSchema=True)
    demographics_df = spark.read.csv("dbfs:/mnt/data/demographics.csv", header=True, inferSchema=True)
    scores_df = spark.read.csv("dbfs:/mnt/data/scores.csv", header=True, inferSchema=True)
    aiml_insights_df = spark.read.csv("dbfs:/mnt/data/aiml_insights.csv", header=True, inferSchema=True)
    return policy_df, claims_df, demographics_df, scores_df, aiml_insights_df

# COMMAND ----------
#
# Function to select specific fields from the demographics DataFrame
def select_fields(demographics_df):
    logger.info("Selecting specific fields from each DataFrame.")
    return demographics_df.select(
        "Customer_ID", "Customer_Name", "Email", "Phone_Number", "Address", "City", "State", "Postal_Code",
        "Date_of_Birth", "Gender", "Marital_Status", "Occupation", "Income_Level", "Customer_Segment"
    )

# COMMAND ----------
#
# Function to join data to consolidate into a comprehensive customer profile
def join_data(selected_demographics_df, policy_df, claims_df):
    logger.info("Joining data to consolidate into a comprehensive customer profile.")
    return selected_demographics_df.join(policy_df, F.col("Customer_ID") == F.col("customer_id"), "inner") \
                                   .join(claims_df, F.col("policy_id") == F.col("Policy_ID"), "inner")

# COMMAND ----------
#
# Function to aggregate claims data to compute metrics
def aggregate_data(joined_df):
    logger.info("Aggregating claims data to compute metrics.")
    return joined_df.groupBy("Customer_ID").agg(
        F.count("Claim_ID").alias("Total_Claims"),
        F.count("policy_id").alias("Policy_Count"),
        F.max("Claim_Date").alias("Recent_Claim_Date"),
        F.avg("Claim_Amount").alias("Average_Claim_Amount")
    )

# COMMAND ----------
#
# Function to implement custom calculations for additional metrics
def custom_calculations(aggregated_df):
    logger.info("Implementing custom calculations for additional metrics.")
    age_expr = F.expr("DATEDIFF(current_date(), to_date(Date_of_Birth, 'yyyy-MM-dd')) / 365")
    claim_to_premium_ratio_expr = F.expr("CASE WHEN total_premium_paid != 0 THEN Claim_Amount / total_premium_paid ELSE 0 END")
    claims_per_policy_expr = F.expr("CASE WHEN Policy_Count != 0 THEN Total_Claims / Policy_Count ELSE 0 END")

    return aggregated_df.withColumn("Age", age_expr) \
                        .withColumn("Claim_To_Premium_Ratio", claim_to_premium_ratio_expr) \
                        .withColumn("Claims_Per_Policy", claims_per_policy_expr) \
                        .withColumn("Retention_Rate", F.lit(0.85)) \
                        .withColumn("Cross_Sell_Opportunities", F.lit("Multi-Policy Discount, Home Coverage Add-on")) \
                        .withColumn("Upsell_Potential", F.lit("Premium Vehicle Coverage"))

# COMMAND ----------
#
# Function to integrate AI/ML insights and customer scores to enhance the customer profile
def enrich_data(final_df, scores_df, aiml_insights_df):
    logger.info("Integrating AI/ML insights and customer scores to enhance the customer profile.")
    return final_df.join(scores_df, "Customer_ID", "inner") \
                   .join(aiml_insights_df, "Customer_ID", "inner")

# COMMAND ----------
#
# Function to write the consolidated customer profile to a Unity Catalog table
def write_output(enriched_df):
    logger.info("Writing the consolidated customer profile to a Unity Catalog table.")
    enriched_df.write.format("delta").mode("overwrite").saveAsTable("catalog.target_db.Customer_360")

# COMMAND ----------
#
# Main ETL process
try:
    # Load data
    policy_df, claims_df, demographics_df, scores_df, aiml_insights_df = load_data()
    
    # Select fields
    selected_demographics_df = select_fields(demographics_df)
    
    # Join data
    joined_df = join_data(selected_demographics_df, policy_df, claims_df)
    
    # Aggregate data
    aggregated_df = aggregate_data(joined_df)
    
    # Custom calculations
    final_df = custom_calculations(aggregated_df)
    
    # Enrich data
    enriched_df = enrich_data(final_df, scores_df, aiml_insights_df)
    
    # Write output
    write_output(enriched_df)

except Exception as e:
    logger.error(f"An error occurred during the ETL process: {e}")
    raise
